# Payroad: All-in-One AI 벤치마크 (기상 · 수요 · 요금산정)

이 노트북은 Payroad 프로젝트의 3대 핵심 AI 파이프라인을 Kaggle GPU 환경에서 **안정적으로 완주**하도록 재작성한 버전입니다.

**[실험 내용]**
1. **기상 예측 (Weather Forecasting):** 인도 5대 도시 10년치 기상 데이터로 LSTM vs 구글 TimesFM 2.5 비교 (rolling-origin 다중 평가)
2. **미래 수요 예측 (Demand Forecasting):** 공항 허브 시계열 수요를 LSTM으로 예측하고 TimesFM과 비교
3. **AI 기반 요금 산정 (ML-based Pricing):** Rule-based 공식을 넘어 기사 수락률을 반영한 XGBoost & DNN 요금 모델 구축

**주요 개선점 (이전 버전 대비)**
- TimesFM **2.5 최신 API** 로 교체 (구버전 `TimesFm(...)` / `forecast([...])` 는 더 이상 동작하지 않음)
- 각 Part를 독립적으로 감싸 **하나가 실패해도 나머지는 계속 실행**
- LSTM vs TimesFM 을 **동일한 여러 평가 구간(rolling-origin)** 에서 공정 비교 → 신뢰도 있는 결과
- GPU/CPU **자동 감지**, seed 고정으로 **재현성** 확보
- 각 단계마다 진행 상황과 소요 시간 출력

**실행 전 필수 설정**
우측 패널 **Settings → Accelerator → GPU T4 x2 (또는 P100)** 선택 후 `Run All`.
> 참고: 이 노트북은 GPU 1개만 사용해도 충분합니다. T4가 1개든 2개든 속도 차이는 거의 없습니다 (분산 학습을 쓰지 않으므로). **P100 1개** 가 오히려 더 빠를 수 있습니다.

**인터넷 설정:** TimesFM 가중치와 Open-Meteo 데이터를 받으려면 우측 패널 **Settings → Internet → On** 이 반드시 켜져 있어야 합니다.

## 0. 설치 및 환경 설정

TimesFM 2.5는 PyTorch 백엔드를 사용합니다. Kaggle에는 PyTorch가 이미 설치되어 있으므로 `timesfm[torch]` 만 추가로 설치합니다.

> 설치에 2~4분 정도 걸릴 수 있습니다.

In [ ]:
# TimesFM 2.5 + 기타 의존성 설치
# (Kaggle에는 torch, tensorflow, xgboost, sklearn 등이 이미 있지만 버전 안전을 위해 명시)
!pip install -q timesfm[torch] openmeteo-requests requests-cache retry-requests
print("설치 완료.")

In [ ]:
import os, time, warnings, sys
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # Kaggle 헤드리스 환경 안전
import matplotlib.pyplot as plt
import seaborn as sns

# 재현성을 위한 seed 고정
SEED = 42
np.random.seed(SEED)

import tensorflow as tf
tf.random.set_seed(SEED)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error

import xgboost as xgb

# --- 하드웨어 감지 ---
gpus = tf.config.list_physical_devices('GPU')
print("TensorFlow:", tf.__version__)
print("TF가 인식한 GPU:", gpus)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch:", torch.__version__, "| TimesFM 사용 디바이스:", DEVICE)
if DEVICE == "cuda":
    print("GPU 이름:", torch.cuda.get_device_name(0))

# TF가 GPU 메모리를 통째로 잡지 않도록 (TimesFM/torch와 공존)
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception as e:
        print("memory_growth 설정 건너뜀:", e)

## 0-1. TimesFM 2.5 모델 로드 (한 번만)

TimesFM은 **사전훈련된 모델**이라 학습이 필요 없습니다. 가중치(~800MB)를 HuggingFace에서 한 번만 내려받아 캐시합니다.

- `max_context`: 모델에 넣을 과거 시점 최대 길이
- `max_horizon`: 예측할 미래 시점 최대 길이

여기서는 기상(168시간 입력 → 24시간 예측)과 수요(168→24)를 모두 커버하도록 넉넉히 잡습니다.

In [ ]:
# TimesFM 2.5 로드 및 컴파일 (실패해도 노트북 전체가 죽지 않도록 방어)
TIMESFM_OK = False
tfm = None
try:
    import timesfm
    torch.set_float32_matmul_precision("high")

    t0 = time.time()
    tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained("google/timesfm-2.5-200m-pytorch")
    tfm.compile(timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=False,   # 기온은 음수 가능 → False
        fix_quantile_crossing=True,
    ))
    TIMESFM_OK = True
    print(f"TimesFM 2.5 로드 완료 ({time.time()-t0:.1f}s)")
except Exception as e:
    print("[경고] TimesFM 로드 실패 - TimesFM 비교는 건너뛰고 LSTM 결과만 사용합니다.")
    print("원인:", repr(e))


def timesfm_forecast(context_series, horizon):
    # 단일 시계열에 대해 TimesFM 점예측 반환. 실패 시 None.
    if not TIMESFM_OK:
        return None
    try:
        arr = np.asarray(context_series, dtype=np.float32)
        point, _q = tfm.forecast(horizon=horizon, inputs=[arr])
        return np.asarray(point[0], dtype=np.float32)
    except Exception as e:
        print("  TimesFM 예측 실패:", repr(e))
        return None

# Part 1: 인도 5대 도시 기상 예측 (LSTM vs TimesFM 2.5)

10년치 시간별 기온 데이터를 수집한 뒤, **동일한 여러 평가 구간(rolling-origin)** 에서 두 모델을 공정하게 비교합니다.

- **입력:** 과거 168시간(7일)
- **예측:** 미래 24시간(1일)
- **평가:** 마지막 구간 여러 개(기본 20개)에 대해 각각 예측하고 MAE/RMSE 평균

LSTM은 각 도시별로 학습하고, TimesFM은 학습 없이 같은 구간을 zero-shot 예측합니다.

In [ ]:
# Part 1-a: 데이터 수집
import openmeteo_requests
import requests_cache
from retry_requests import retry

cache_session = requests_cache.CachedSession('.cache', expire_after=-1)
retry_session = retry(cache_session, retries=5, backoff_factor=0.5)
openmeteo = openmeteo_requests.Client(session=retry_session)

cities = {
    "Bengaluru": {"lat": 12.9716, "lon": 77.5946},
    "Mumbai":    {"lat": 19.0760, "lon": 72.8777},
    "New Delhi": {"lat": 28.6139, "lon": 77.2090},
    "Chennai":   {"lat": 13.0827, "lon": 80.2707},
    "Kolkata":   {"lat": 22.5726, "lon": 88.3639},
}

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
city_dfs = {}

print("인도 5대 도시 10년치 시간별 기온 수집 중...")
for city, coords in cities.items():
    try:
        params = {
            "latitude": coords["lat"], "longitude": coords["lon"],
            "start_date": "2014-01-01", "end_date": "2024-01-01",
            "hourly": "temperature_2m",
        }
        resp = openmeteo.weather_api(ARCHIVE_URL, params=params)[0]
        temps = resp.Hourly().Variables(0).ValuesAsNumpy()
        s = pd.Series(temps).interpolate().dropna().astype(np.float32)
        city_dfs[city] = s.values
        print(f"  [{city}] {len(s):,} 시간 수집 완료")
    except Exception as e:
        print(f"  [{city}] 수집 실패 - 건너뜀:", repr(e))

print("수집된 도시 수:", len(city_dfs))

In [ ]:
# Part 1-b: 공정 비교 함수 (rolling-origin 평가)
LOOK_BACK = 168        # 입력 길이 (7일)
HORIZON   = 24         # 예측 길이 (1일)
N_EVAL    = 20         # 평가 구간 개수 (많을수록 신뢰도↑, 시간↑)
EVAL_STRIDE = 24       # 평가 구간 간격 (24 = 하루씩 이동)


def make_supervised(series, look_back, horizon, stride=1):
    # 슬라이딩 윈도우로 (X, Y) 생성. stride로 샘플 수 조절.
    X, Y = [], []
    last = len(series) - look_back - horizon
    for i in range(0, last, stride):
        X.append(series[i:i+look_back])
        Y.append(series[i+look_back:i+look_back+horizon])
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)


def build_lstm(look_back, horizon):
    m = Sequential([
        LSTM(128, return_sequences=True, input_shape=(look_back, 1)),
        Dropout(0.2),
        LSTM(64),
        Dropout(0.2),
        Dense(horizon),
    ])
    m.compile(optimizer='adam', loss='mse')
    return m

In [ ]:
# Part 1-c: 도시별 학습 및 LSTM vs TimesFM 평가
weather_results = {}

for city, series in city_dfs.items():
    print(f"\n===== [{city}] =====")
    t0 = time.time()

    # 평가 구간(마지막 N_EVAL개)을 학습에서 제외
    eval_span = LOOK_BACK + HORIZON + (N_EVAL - 1) * EVAL_STRIDE
    if len(series) <= eval_span + LOOK_BACK:
        print("  데이터가 부족하여 건너뜀")
        continue
    train_series = series[:-eval_span]

    # --- 학습 데이터 구성 (stride로 샘플 수 제어: 정확도/시간 균형) ---
    Xtr, Ytr = make_supervised(train_series, LOOK_BACK, HORIZON, stride=6)
    scaler = MinMaxScaler()
    scaler.fit(train_series.reshape(-1, 1))
    Xtr_s = scaler.transform(Xtr.reshape(-1, 1)).reshape(Xtr.shape)[..., None]
    Ytr_s = scaler.transform(Ytr.reshape(-1, 1)).reshape(Ytr.shape)

    # --- LSTM 학습 ---
    lstm = build_lstm(LOOK_BACK, HORIZON)
    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    lstm.fit(Xtr_s, Ytr_s, batch_size=256, epochs=30,
             validation_split=0.1, callbacks=[es], verbose=0)
    print(f"  LSTM 학습 완료 (샘플 {len(Xtr):,}개, {time.time()-t0:.1f}s)")

    # --- rolling-origin 평가 ---
    lstm_maes, tfm_maes = [], []
    lstm_rmses, tfm_rmses = [], []
    base = len(series) - eval_span
    for k in range(N_EVAL):
        s = base + k * EVAL_STRIDE
        past   = series[s:s+LOOK_BACK]
        future = series[s+LOOK_BACK:s+LOOK_BACK+HORIZON]

        # LSTM 예측
        past_s = scaler.transform(past.reshape(-1, 1)).reshape(1, LOOK_BACK, 1)
        lp = scaler.inverse_transform(lstm.predict(past_s, verbose=0))[0]
        lstm_maes.append(mean_absolute_error(future, lp))
        lstm_rmses.append(np.sqrt(mean_squared_error(future, lp)))

        # TimesFM 예측 (zero-shot)
        tp = timesfm_forecast(past, HORIZON)
        if tp is not None:
            tfm_maes.append(mean_absolute_error(future, tp))
            tfm_rmses.append(np.sqrt(mean_squared_error(future, tp)))

    weather_results[city] = {
        "LSTM_MAE": float(np.mean(lstm_maes)),
        "LSTM_RMSE": float(np.mean(lstm_rmses)),
        "TimesFM_MAE": float(np.mean(tfm_maes)) if tfm_maes else np.nan,
        "TimesFM_RMSE": float(np.mean(tfm_rmses)) if tfm_maes else np.nan,
    }
    r = weather_results[city]
    print(f"  [결과] LSTM  MAE {r['LSTM_MAE']:.2f} / RMSE {r['LSTM_RMSE']:.2f}")
    print(f"         TimesFM MAE {r['TimesFM_MAE']:.2f} / RMSE {r['TimesFM_RMSE']:.2f}")
    print(f"  총 소요 {time.time()-t0:.1f}s")

In [ ]:
# Part 1-d: 결과 요약 및 시각화
if weather_results:
    wdf = pd.DataFrame(weather_results).T
    print("=== 기상 예측 정확도 (MAE, °C, 낮을수록 좋음) ===")
    print(wdf.round(2))

    ax = wdf[["LSTM_MAE", "TimesFM_MAE"]].plot(kind="bar", figsize=(10, 5))
    ax.set_ylabel("MAE (°C)")
    ax.set_title("도시별 기온 예측 정확도: LSTM vs TimesFM 2.5")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig("weather_comparison.png", dpi=130)
    plt.show()
    print("그래프 저장: weather_comparison.png")
else:
    print("기상 결과 없음 - Part 1을 건너뛰었습니다.")

# Part 2: 공항 허브 미래 수요(Demand) 예측

과거 택시 호출량 패턴(출퇴근 주기, 주중/주말)을 학습해 1시간 뒤 수요를 예측합니다.
실제 데이터가 없으므로 **현실적인 패턴을 합성**하고, LSTM과 TimesFM을 같은 테스트 구간에서 비교합니다.

> 실제 프로젝트에서는 이 합성 데이터를 실측 호출 로그로 교체하면 그대로 동작합니다.

In [ ]:
# Part 2-a: 합성 수요 데이터 생성
np.random.seed(SEED)
N_HOURS = 43800  # 약 5년
hours = np.arange(N_HOURS)
daily   = 50 * np.sin(2*np.pi*hours/24)          # 하루 주기
weekly  = 20 * np.sin(2*np.pi*hours/(24*7))       # 주간 주기
trend   = np.linspace(0, 15, N_HOURS)             # 완만한 성장
noise   = np.random.normal(0, 8, N_HOURS)
demand  = np.maximum(0, 100 + daily + weekly + trend + noise).astype(np.float32)
print("수요 시계열 생성:", demand.shape)

In [ ]:
# Part 2-b: 수요 LSTM 학습 + TimesFM 비교 (rolling-origin)
demand_results = {}
try:
    D_LOOK, D_HORIZON = 168, 24
    D_NEVAL, D_STRIDE = 20, 24

    eval_span = D_LOOK + D_HORIZON + (D_NEVAL - 1) * D_STRIDE
    d_train = demand[:-eval_span]

    Xd, Yd = make_supervised(d_train, D_LOOK, D_HORIZON, stride=6)
    d_scaler = MinMaxScaler()
    d_scaler.fit(d_train.reshape(-1, 1))
    Xd_s = d_scaler.transform(Xd.reshape(-1,1)).reshape(Xd.shape)[..., None]
    Yd_s = d_scaler.transform(Yd.reshape(-1,1)).reshape(Yd.shape)

    t0 = time.time()
    d_lstm = build_lstm(D_LOOK, D_HORIZON)
    es = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
    d_lstm.fit(Xd_s, Yd_s, batch_size=256, epochs=30,
               validation_split=0.1, callbacks=[es], verbose=0)
    print(f"수요 LSTM 학습 완료 (샘플 {len(Xd):,}개, {time.time()-t0:.1f}s)")

    l_maes, t_maes = [], []
    base = len(demand) - eval_span
    last_example = None
    for k in range(D_NEVAL):
        s = base + k * D_STRIDE
        past   = demand[s:s+D_LOOK]
        future = demand[s+D_LOOK:s+D_LOOK+D_HORIZON]
        past_s = d_scaler.transform(past.reshape(-1,1)).reshape(1, D_LOOK, 1)
        lp = d_scaler.inverse_transform(d_lstm.predict(past_s, verbose=0))[0]
        l_maes.append(mean_absolute_error(future, lp))
        tp = timesfm_forecast(past, D_HORIZON)
        if tp is not None:
            t_maes.append(mean_absolute_error(future, tp))
        if k == D_NEVAL - 1:
            last_example = (past, future, lp, tp)

    demand_results = {
        "LSTM_MAE": float(np.mean(l_maes)),
        "TimesFM_MAE": float(np.mean(t_maes)) if t_maes else np.nan,
    }
    print("=== 수요 예측 정확도 (MAE, 낮을수록 좋음) ===")
    print(demand_results)

    # 마지막 구간 시각화
    if last_example:
        past, future, lp, tp = last_example
        plt.figure(figsize=(11, 4))
        plt.plot(range(D_HORIZON), future, label="Actual", marker="o")
        plt.plot(range(D_HORIZON), lp, label="LSTM", linestyle="--")
        if tp is not None:
            plt.plot(range(D_HORIZON), tp, label="TimesFM", linestyle=":")
        plt.title("공항 수요 예측 (마지막 24시간 구간)")
        plt.xlabel("시간(+h)"); plt.ylabel("호출 수"); plt.legend()
        plt.tight_layout(); plt.savefig("demand_forecast.png", dpi=130); plt.show()
        print("그래프 저장: demand_forecast.png")
except Exception as e:
    print("[경고] Part 2 실패 - 건너뜁니다:", repr(e))

# Part 3: AI 기반 최적 요금 산정 모델 (XGBoost & DNN)

현재 시스템의 `Base Fare × Alpha × Beta` 는 사람이 만든 단순 규칙입니다.
여기서는 **100만 건의 가상 주행 데이터**(기사 수락/거절 이력 포함)를 생성하고,
**기사가 실제로 수락한 요금 수준**을 XGBoost와 DNN이 학습하여
"수락될 가능성이 높으면서 합리적인 요금"을 산출하도록 합니다.

> 즉, 룰 기반 요금을 정답이 아니라 *후보* 로 보고, 시장(수락 여부)이 걸러낸 요금을 학습합니다.

In [ ]:
# Part 3-a: 시뮬레이션 데이터 생성
np.random.seed(SEED)
N = 1_000_000
demand_f   = np.random.randint(0, 100, N).astype(np.float32)
supply_f   = np.random.randint(1, 50, N).astype(np.float32)
rain_f     = np.random.uniform(0, 50, N).astype(np.float32)
distance_f = np.random.uniform(1, 40, N).astype(np.float32)

alpha = np.clip(demand_f / supply_f, 0.7, 3.0)
beta  = 1.0 + (rain_f / 50) * 0.5
base_fare = 30 + 15 * distance_f
rule_fare = base_fare * alpha * beta

# 기사 수락 확률(비선형): 비 많은데 요금 낮으면↓, 공급 많으면↑
logit = (rule_fare / base_fare) - 1.5 - (rain_f/20) + (supply_f/np.maximum(demand_f,1))
accept_prob = 1 / (1 + np.exp(-logit))
accepted = np.random.binomial(1, np.clip(accept_prob, 0.05, 0.95))

pricing_df = pd.DataFrame({
    "demand": demand_f, "supply": supply_f, "rain": rain_f,
    "distance": distance_f, "rule_fare": rule_fare, "accepted": accepted,
})
print("시뮬레이션 데이터:", pricing_df.shape)
print("수락률: {:.1%}".format(pricing_df['accepted'].mean()))

In [ ]:
# Part 3-b: XGBoost & DNN 학습 (수락된 건의 요금 학습)
pricing_results = {}
try:
    train_df = pricing_df[pricing_df["accepted"] == 1]
    feats = ["demand", "supply", "rain", "distance"]
    Xp, yp = train_df[feats].values, train_df["rule_fare"].values
    Xtr, Xte, ytr, yte = train_test_split(Xp, yp, test_size=0.2, random_state=SEED)

    # XGBoost (GPU 가속: tree_method=hist + device=cuda)
    t0 = time.time()
    xgb_params = dict(n_estimators=400, learning_rate=0.05, max_depth=7,
                      subsample=0.8, colsample_bytree=0.8, n_jobs=-1)
    if DEVICE == "cuda":
        xgb_params.update(tree_method="hist", device="cuda")
    xgb_model = xgb.XGBRegressor(**xgb_params)
    xgb_model.fit(Xtr, ytr)
    xgb_pred = xgb_model.predict(Xte)
    pricing_results["XGBoost_MAE"] = float(mean_absolute_error(yte, xgb_pred))
    print(f"XGBoost 학습 완료 ({time.time()-t0:.1f}s) | MAE {pricing_results['XGBoost_MAE']:.2f}")

    # DNN
    t0 = time.time()
    sc = StandardScaler().fit(Xtr)
    Xtr_s, Xte_s = sc.transform(Xtr), sc.transform(Xte)
    dnn = Sequential([
        Dense(128, activation="relu", input_dim=4),
        BatchNormalization(), Dropout(0.2),
        Dense(64, activation="relu"),
        Dense(32, activation="relu"),
        Dense(1, activation="linear"),
    ])
    dnn.compile(optimizer="adam", loss="mse")
    es = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)
    dnn.fit(Xtr_s, ytr, batch_size=2048, epochs=25,
            validation_split=0.1, callbacks=[es], verbose=0)
    dnn_pred = dnn.predict(Xte_s, verbose=0).flatten()
    pricing_results["DNN_MAE"] = float(mean_absolute_error(yte, dnn_pred))
    print(f"DNN 학습 완료 ({time.time()-t0:.1f}s) | MAE {pricing_results['DNN_MAE']:.2f}")

    # 샘플 비교표
    sample = pd.DataFrame(Xte[:8], columns=feats)
    sample["Rule-based (기존)"] = yte[:8]
    sample["XGBoost (AI)"] = xgb_model.predict(Xte[:8])
    sample["DNN (AI)"] = dnn.predict(sc.transform(Xte[:8]), verbose=0).flatten()
    print("\n=== 요금 비교 (샘플 8건) ===")
    print(sample.round(2).to_string(index=False))
except Exception as e:
    print("[경고] Part 3 실패 - 건너뜁니다:", repr(e))

# 최종 요약

세 파이프라인의 결과를 한 곳에 모읍니다. (일부 Part가 실패해도 나머지는 표시됩니다.)

In [ ]:
print("="*55)
print(" PAYROAD AI 벤치마크 최종 요약")
print("="*55)

print("\n[Part 1] 기상 예측 (MAE, °C)")
if weather_results:
    print(pd.DataFrame(weather_results).T[["LSTM_MAE", "TimesFM_MAE"]].round(2).to_string())
else:
    print("  (결과 없음)")

print("\n[Part 2] 수요 예측 (MAE)")
print(" ", demand_results if demand_results else "(결과 없음)")

print("\n[Part 3] 요금 산정 (MAE)")
print(" ", pricing_results if pricing_results else "(결과 없음)")

print("\n생성된 그래프 파일:")
for f in ["weather_comparison.png", "demand_forecast.png"]:
    print("  -", f, "(존재)" if os.path.exists(f) else "(없음)")
print("\n완료.")